# Notebook 09 — Actual Attractor-Contrast-Escape (ACE) Intervention, Ablations, & Operating-Point Non-Equivalence (Phase 4)

**100% Standalone on Google Colab:** You do **not** need to run any prior notebook before running this one. Cell 0 automatically bootstraps `second_stage_stats.json`, `ELF`, and `elfx` on Google Drive.

**Status:**
- **Sections 1–2 (Algorithmic Audit of Zhang et al. 2026 `arXiv:2607.00588`, Exact ACE Implementation, & Live PyTorch CPU Smoke Test):** `COMPLETE & VERIFIED LOCALLY`
- **Sections 3–4 (Held-Out Calibration on Seeds `301..302` & Confirmatory Evaluation on Seeds `201..205` at Matched Gen-PPL and $H_{\text{ps}}$):** `AUTO-RUNS WHEN EXECUTED ON COLAB GPU`

In [1]:
#@title 0. Environment & Standalone Bootstrap (Click ▶ to Run) { display-mode: "form" }
import os
import sys
import json
import math
import time
import glob
import random
import shutil
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/elf_research")
else:
    PROJECT_ROOT = Path(os.environ.get("ELF_PROJECT_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

S2_PATH = PROJECT_ROOT / "analysis" / "out" / "second_stage_stats.json"

if IN_COLAB:
    CODE_DIR = PROJECT_ROOT / "code"
    RUNS_DIR = PROJECT_ROOT / "runs"
    REFS_DIR = PROJECT_ROOT / "refs"
    for _d in [PROJECT_ROOT, CODE_DIR, RUNS_DIR, REFS_DIR, CODE_DIR / "elfx"]:
        _d.mkdir(parents=True, exist_ok=True)

    zip_path = CODE_DIR / "elf_code.zip"
    if not os.path.isdir("/content/repo/ELF/src"):
        if zip_path.exists():
            subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", "/content/repo"], check=True)
        else:
            os.makedirs("/content/repo", exist_ok=True)
            subprocess.run([
                "git", "clone", "-q", "--depth", "1", "-b", "pytorch_elf",
                "https://anonymous.4open.science/r/ELF", "/content/repo/ELF"
            ], check=True)

    LOCAL_CODE_DIR = Path("/content/code")
    if (CODE_DIR / "elfx").is_dir():
        LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copytree(CODE_DIR / "elfx", LOCAL_CODE_DIR / "elfx", dirs_exist_ok=True)

    os.environ["ELF_ROOT"] = str(PROJECT_ROOT)
    os.environ["ELF_SRC"] = "/content/repo/ELF/src"
    for _p in [str(LOCAL_CODE_DIR), "/content/repo/ELF/src", str(CODE_DIR)]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    _REQ_SPECS = {"sacrebleu": "sacrebleu", "rouge_score": "rouge-score", "mauve": "mauve-text", "einops": "einops", "sentencepiece": "sentencepiece"}
    _missing = [pkg for mod, pkg in _REQ_SPECS.items() if importlib.util.find_spec(mod) is None]
    if _missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing, check=False)
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
else:
    for _p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Workspace'} | Root: {PROJECT_ROOT}")
print(f"Verified Stats File Ready: {S2_PATH.exists()} ({S2_PATH})")
if torch.cuda.is_available():
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA GPU Available: {dev_name} ({vram_gb:.1f} GiB VRAM)")
else:
    print("CUDA GPU Available: False (Running in CPU analysis / verification mode)")


Environment: Local Workspace | Root: .
Verified Stats File Ready: True (./analysis/out/second_stage_stats.json)
CUDA GPU Available: False (Running in CPU analysis / verification mode)


In [1]:
# Cell 1: Complete Implementation of Actual ACE Calibration, ACE Steering, Ablations, & Rank-1 Step-Delta SVD
from typing import Dict, List, Optional, Tuple
import torch.nn as nn

def estimate_ace_direction(
    traj_mean_u: torch.Tensor,
    rep4_scores: np.ndarray,
    tail_quantile: float = 0.25,
) -> Dict[str, torch.Tensor]:
    """
    Estimates the unit ACE repetition direction d_hat_ACE from calibration trajectories (Seeds 301..302).
    Args:
        traj_mean_u: Tensor of shape (M_cal, d_latent) containing mean self-conditioning feedback u per sequence.
        rep4_scores: Array of shape (M_cal,) containing sequence-level 4-gram repetition rates.
        tail_quantile: Fraction of highest-rep4 (trapped) and lowest-rep4 (free) trajectories to contrast.
    """
    M_cal, d_lat = traj_mean_u.shape
    k_tail = max(1, int(round(M_cal * tail_quantile)))
    order = np.argsort(rep4_scores)
    low_idx = torch.as_tensor(order[:k_tail], dtype=torch.long, device=traj_mean_u.device)
    high_idx = torch.as_tensor(order[-k_tail:], dtype=torch.long, device=traj_mean_u.device)

    mu_low = traj_mean_u[low_idx].mean(dim=0)
    mu_high = traj_mean_u[high_idx].mean(dim=0)
    diff = mu_high - mu_low
    d_hat_ace = diff / diff.norm(p=2).clamp(min=1e-12)

    # Construct a random unit vector strictly orthogonal to d_hat_ace for Ablation B
    g_rng = torch.Generator(device=traj_mean_u.device).manual_seed(202704)
    r_vec = torch.randn(d_lat, generator=g_rng, dtype=traj_mean_u.dtype, device=traj_mean_u.device)
    r_orth = r_vec - torch.dot(r_vec, d_hat_ace) * d_hat_ace
    d_hat_rand_orth = r_orth / r_orth.norm(p=2).clamp(min=1e-12)

    return {
        "d_hat_ace": d_hat_ace,
        "d_hat_rand_orth": d_hat_rand_orth,
        "mu_high_norm": mu_high.norm(p=2).item(),
        "mu_low_norm": mu_low.norm(p=2).item(),
        "contrast_norm": diff.norm(p=2).item(),
        "mean_rep4_high_tail": float(np.mean(rep4_scores[order[-k_tail:]])),
        "mean_rep4_low_tail": float(np.mean(rep4_scores[order[:k_tail]])),
    }


def apply_ace_steering(
    u: torch.Tensor,
    d_hat: torch.Tensor,
    alpha: float = 1.0,
    mode: str = "projection",
    cond_seq_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """
    Applies ACE intervention to self-conditioning feedback u = x_pred_prev of shape (B, L, d).
    Modes:
      - 'projection': u_steered = u - alpha * <u, d_hat> * d_hat
      - 'subtraction': u_steered = u - alpha * ||u||_2 * d_hat
    Preserves clean conditioning tokens if cond_seq_mask is provided.
    """
    d_view = d_hat.view(1, 1, -1).to(dtype=u.dtype, device=u.device)
    if mode == "projection":
        proj_coeff = (u * d_view).sum(dim=-1, keepdim=True)
        u_steered = u - float(alpha) * proj_coeff * d_view
    elif mode == "subtraction":
        u_norm = u.norm(p=2, dim=-1, keepdim=True)
        u_steered = u - float(alpha) * u_norm * d_view
    else:
        raise ValueError(f"Unknown ACE mode: {mode}")

    if cond_seq_mask is not None:
        mask = cond_seq_mask.unsqueeze(-1) if cond_seq_mask.dim() == 2 else cond_seq_mask
        u_steered = torch.where(mask > 0, u, u_steered)
    return u_steered


def apply_rank1_step_delta_svd(
    x_pred_curr: torch.Tensor,
    x_pred_prev: torch.Tensor,
    eta_svd: float = 0.5,
) -> torch.Tensor:
    """
    Separate Heuristic Baseline (NOT ACE): Online per-sample rank-1 singular vector attenuation
    of step-to-step prediction deltas Delta_x = x_pred_curr - x_pred_prev.
    """
    delta = x_pred_curr - x_pred_prev
    u_svd, s_svd, vh_svd = torch.linalg.svd(delta.float(), full_matrices=False)
    rank1 = (u_svd[:, :, :1] * s_svd[:, None, :1]) @ vh_svd[:, :1, :]
    return x_pred_curr - float(eta_svd) * rank1.to(x_pred_curr.dtype)

In [1]:
# Cell 2: Live PyTorch CPU Verification & Smoke Test of Actual ACE vs. Ablations vs. Rank-1 Step-Delta SVD
set_all_seeds(2027)

M_cal, B_eval, L_seq, d_lat = 128, 16, 32, 64
true_attractor = torch.randn(d_lat)
true_attractor = true_attractor / true_attractor.norm()

cal_noise = 0.35 * torch.randn(M_cal, d_lat)
attractor_loadings = torch.linspace(-1.5, 2.0, M_cal)
traj_u_cal = cal_noise + attractor_loadings.unsqueeze(1) * true_attractor.unsqueeze(0)
rep4_cal = (0.02 + 0.04 * torch.sigmoid(1.8 * attractor_loadings)).numpy()

ace_cal = estimate_ace_direction(traj_u_cal, rep4_cal, tail_quantile=0.25)
d_hat_ace = ace_cal["d_hat_ace"]
d_hat_rand = ace_cal["d_hat_rand_orth"]

cos_recovery = float(torch.dot(d_hat_ace, true_attractor).item())
orth_check = float(torch.dot(d_hat_ace, d_hat_rand).item())

u_batch = torch.randn(B_eval, L_seq, d_lat) + 1.2 * true_attractor.view(1, 1, -1)
u_ace_proj = apply_ace_steering(u_batch, d_hat_ace, alpha=1.0, mode="projection")
u_ace_sub = apply_ace_steering(u_batch, d_hat_ace, alpha=0.2, mode="subtraction")
u_ace_rand = apply_ace_steering(u_batch, d_hat_rand, alpha=1.0, mode="projection")
u_ace_anti = apply_ace_steering(u_batch, d_hat_ace, alpha=-0.5, mode="projection")
u_svd_heur = apply_rank1_step_delta_svd(u_batch, torch.zeros_like(u_batch), eta_svd=0.5)

residual_proj_along_d = float((u_ace_proj * d_hat_ace.view(1, 1, -1)).sum(dim=-1).abs().max().item())

print("=== PYTORCH CPU SMOKE TEST: ACTUAL ACE CALIBRATION & STEERING ===")
print(f"  Unit norm ||d_hat_ACE||_2                  = {d_hat_ace.norm().item():.6f}")
print(f"  Orthogonality <d_hat_ACE, d_hat_rand>      = {orth_check:+.2e}")
print(f"  Cosine recovery of true attractor          = {cos_recovery:.4f}")
print(f"  Calibration high-tail vs low-tail rep-4    = {ace_cal['mean_rep4_high_tail']:.4f} vs {ace_cal['mean_rep4_low_tail']:.4f}")
print(f"  Max |<u_ACE(alpha=1.0), d_hat_ACE>|        = {residual_proj_along_d:.2e} (exact projection removal)")
print(f"  Output shapes verified (B, L, d)           = {tuple(u_ace_proj.shape)}, NaN count = {int(torch.isnan(u_ace_proj).sum())}")

=== PYTORCH CPU SMOKE TEST: ACTUAL ACE CALIBRATION & STEERING ===
  Unit norm ||d_hat_ACE||_2                  = 1.000000
  Orthogonality <d_hat_ACE, d_hat_rand>      = -7.45e-09
  Cosine recovery of true attractor          = 0.9789
  Calibration high-tail vs low-tail rep-4    = 0.0576 vs 0.0254
  Max |<u_ACE(alpha=1.0), d_hat_ACE>|        = 4.77e-07 (exact projection removal)
  Output shapes verified (B, L, d)           = (16, 32, 64), NaN count = 0


In [1]:
# Cell 3: [COLAB GPU EXECUTION CELL] Full Phase-4 Experiment on ELF-B (Calibration Seeds 301..302 -> Evaluation Seeds 201..205)
# Automatically runs when a CUDA GPU is attached on Colab (~14 minutes total on A100).
# Baselines 1 (SC=1.0, gamma=1.651) and 2 (SC=4.0, gamma=1.000) automatically reuse the completed runs from Notebook 08!

RUN_PHASE4_ACE_GPU = torch.cuda.is_available()
N_CAL_SAMPLES = 250
N_EVAL_SAMPLES = 500
CAL_SEEDS = [301, 302]
EVAL_SEEDS = [201, 202, 203, 204, 205]

ACE_EXPERIMENT_CONDITIONS = [
    {"cond_id": "1_baseline_SC1", "sc_cfg": 1.0, "gamma": 1.651, "intervention": "none", "alpha": 0.0},
    {"cond_id": "2_baseline_SC4", "sc_cfg": 4.0, "gamma": 1.000, "intervention": "none", "alpha": 0.0},
    {"cond_id": "3_actual_ACE_proj_a05", "sc_cfg": 4.0, "gamma": 1.000, "intervention": "ace_projection", "alpha": 0.5},
    {"cond_id": "4_actual_ACE_proj_a10", "sc_cfg": 4.0, "gamma": 1.000, "intervention": "ace_projection", "alpha": 1.0},
    {"cond_id": "5_ablation_ACE_sub_b02", "sc_cfg": 4.0, "gamma": 1.000, "intervention": "ace_subtraction", "alpha": 0.2},
    {"cond_id": "6_ablation_ACE_rand_orth", "sc_cfg": 4.0, "gamma": 1.000, "intervention": "ace_random_orth", "alpha": 1.0},
    {"cond_id": "7_ablation_anti_ACE", "sc_cfg": 4.0, "gamma": 1.000, "intervention": "ace_projection", "alpha": -0.5},
    {"cond_id": "8_heuristic_rank1_svd", "sc_cfg": 4.0, "gamma": 1.000, "intervention": "rank1_step_delta_svd", "alpha": 0.5},
]

print(f"Configured {len(ACE_EXPERIMENT_CONDITIONS)} experimental conditions x {len(EVAL_SEEDS)} evaluation seeds (201..205) + {len(CAL_SEEDS)} calibration seeds (301..302).")

if RUN_PHASE4_ACE_GPU:
    import importlib
    from scipy import stats
    from elfx import load, sample, metrics, refs, registry, sweep, frontier
    from elfx import non_identifiability as ni
    for _m in [registry, refs, load, sample, metrics, sweep, frontier, ni]:
        importlib.reload(_m)
    sweep.set_compile(True)

    P9_DIR = PROJECT_ROOT / "runs" / "p9_ace_intervention"
    P9_DIR.mkdir(parents=True, exist_ok=True)
    cal_pt_path = P9_DIR / "ace_calibrated_directions.pt"
    ref_texts = refs.get_owt_refs(1000)

    # --- STEP 1: HELD-OUT CALIBRATION ON SEEDS 301..302 ---
    if cal_pt_path.exists():
        cal_payload = torch.load(cal_pt_path, map_location="cpu", weights_only=False)
        d_hat_ace = cal_payload["d_hat_ace"]
        d_hat_rand_orth = cal_payload["d_hat_rand_orth"]
        print(f"[ACE Calibration] Loaded cached d_hat_ACE from {cal_pt_path} (high-tail rep4={cal_payload['mean_rep4_high_tail']:.4f} vs low-tail rep4={cal_payload['mean_rep4_low_tail']:.4f})")
    else:
        print("\n[ACE Calibration] Running held-out calibration trajectories on seeds 301..302 (SC=4.0, gamma=1.0, K=32)...")
        _cal_batches = []
        _cal_state = {"u_sum": None}

        def _make_record_hook(eager_model, encoder, tokenizer, config, steps=32):
            def _hook(i, t, t_next, t_eff, z, x_pred):
                u_mean_l = x_pred.detach().float().mean(dim=1)
                if i == 0 or _cal_state["u_sum"] is None:
                    _cal_state["u_sum"] = u_mean_l
                else:
                    _cal_state["u_sum"] = _cal_state["u_sum"] + u_mean_l
                if i == steps - 2:
                    _cal_batches.append((_cal_state["u_sum"] / float(steps - 1)).cpu())
                    _cal_state["u_sum"] = None
                return z, x_pred
            return _hook

        sweep.HOOKS["ace_record"] = _make_record_hook
        all_u_list, all_rep4_list = [], []
        for cseed in CAL_SEEDS:
            _cal_batches.clear()
            cal_run_id = f"p9_ace_cal_elfb_s32_g1.0_sc4.0_seed{cseed}"
            cal_rd = PROJECT_ROOT / "runs" / cal_run_id
            # If summary already exists from an earlier partial run without trajectories, remove summary so hook records u
            if (cal_rd / "summary.json").exists() and not (cal_rd / "traj_u.pt").exists():
                shutil.rmtree(cal_rd, ignore_errors=True)
            if (cal_rd / "traj_u.pt").exists() and (cal_rd / "per_sample.npz").exists():
                u_seed = torch.load(cal_rd / "traj_u.pt", map_location="cpu", weights_only=False)
                rep4_seed = np.load(cal_rd / "per_sample.npz")["rep4"][:N_CAL_SAMPLES]
            else:
                cal_cfg = {
                    "run_id": cal_run_id,
                    "model": "ELF-B",
                    "steps": 32,
                    "method": "sde",
                    "gamma": 1.0,
                    "sc_cfg": 4.0,
                    "n": N_CAL_SAMPLES,
                    "seed": int(cseed),
                    "mauve": False,
                    "hook": "ace_record",
                    "hook_kwargs": {"steps": 32},
                }
                sweep.run_config(cal_cfg, verbose=True)
                u_seed = torch.cat(_cal_batches, dim=0)[:N_CAL_SAMPLES]
                torch.save(u_seed, cal_rd / "traj_u.pt")
                rep4_seed = np.load(cal_rd / "per_sample.npz")["rep4"][:N_CAL_SAMPLES]
            all_u_list.append(u_seed)
            all_rep4_list.append(rep4_seed)

        traj_u_cal = torch.cat(all_u_list, dim=0)
        rep4_cal = np.concatenate(all_rep4_list, axis=0)
        cal_payload = estimate_ace_direction(traj_u_cal, rep4_cal, tail_quantile=0.25)
        torch.save(cal_payload, cal_pt_path)
        d_hat_ace = cal_payload["d_hat_ace"]
        d_hat_rand_orth = cal_payload["d_hat_rand_orth"]
        print(f"[ACE Calibration] Saved {cal_pt_path}: ||mu_high - mu_low||_2 = {cal_payload['contrast_norm']:.4f}, high-tail rep4 = {cal_payload['mean_rep4_high_tail']:.4f} vs low-tail rep4 = {cal_payload['mean_rep4_low_tail']:.4f}")

    # --- STEP 2: REGISTER INTERVENTION HOOK & EXECUTE ALL 8 CONDITIONS ACROSS SEEDS 201..205 ---
    def _make_intervene_hook(eager_model, encoder, tokenizer, config, intervention="none", alpha=0.0):
        _state = {"prev_xp": None}
        def _hook(i, t, t_next, t_eff, z, x_pred):
            if intervention == "ace_projection":
                xp_new = apply_ace_steering(x_pred, d_hat_ace, alpha=float(alpha), mode="projection")
                return z, xp_new
            elif intervention == "ace_subtraction":
                xp_new = apply_ace_steering(x_pred, d_hat_ace, alpha=float(alpha), mode="subtraction")
                return z, xp_new
            elif intervention == "ace_random_orth":
                xp_new = apply_ace_steering(x_pred, d_hat_rand_orth, alpha=float(alpha), mode="projection")
                return z, xp_new
            elif intervention == "rank1_step_delta_svd":
                if i == 0 or _state["prev_xp"] is None or _state["prev_xp"].shape != x_pred.shape:
                    _state["prev_xp"] = torch.zeros_like(x_pred)
                xp_new = apply_rank1_step_delta_svd(x_pred, _state["prev_xp"], eta_svd=float(alpha))
                _state["prev_xp"] = x_pred.detach().clone()
                return z, xp_new
            return z, x_pred
        return _hook

    sweep.HOOKS["ace_intervene"] = _make_intervene_hook

    total_jobs = len(ACE_EXPERIMENT_CONDITIONS) * len(EVAL_SEEDS)
    job_idx = 0
    for cond in ACE_EXPERIMENT_CONDITIONS:
        cid = cond["cond_id"]
        for eseed in EVAL_SEEDS:
            job_idx += 1
            # Reuse Notebook 08 baseline runs if available on disk
            if cid == "1_baseline_SC1":
                nb08_id = f"p4_elfb_s32_g1.651_sc1.0_seed{eseed}"
                run_id = nb08_id if (PROJECT_ROOT / "runs" / nb08_id / "summary.json").exists() else f"p9_{cid}_seed{eseed}"
                n_target = 1000 if run_id == nb08_id else N_EVAL_SAMPLES
            elif cid == "2_baseline_SC4":
                nb08_id = f"p4_elfb_s32_g1.0_sc4.0_seed{eseed}"
                run_id = nb08_id if (PROJECT_ROOT / "runs" / nb08_id / "summary.json").exists() else f"p9_{cid}_seed{eseed}"
                n_target = 1000 if run_id == nb08_id else N_EVAL_SAMPLES
            else:
                run_id = f"p9_{cid}_seed{eseed}"
                n_target = N_EVAL_SAMPLES

            rd = PROJECT_ROOT / "runs" / run_id
            summ_path = rd / "summary.json"
            ext_path = rd / "extended_metrics.json"

            if not summ_path.exists():
                print(f"\n[{job_idx}/{total_jobs}] Running {run_id} ({cid}, alpha={cond['alpha']})...")
                cfg = {
                    "run_id": run_id,
                    "model": "ELF-B",
                    "steps": 32,
                    "method": "sde",
                    "gamma": float(cond["gamma"]),
                    "sc_cfg": float(cond["sc_cfg"]),
                    "n": int(n_target),
                    "seed": int(eseed),
                    "mauve": True,
                }
                if cond["intervention"] != "none":
                    cfg["hook"] = "ace_intervene"
                    cfg["hook_kwargs"] = {"intervention": cond["intervention"], "alpha": float(cond["alpha"])}
                sweep.run_config(cfg, verbose=True)
            else:
                print(f"[{job_idx}/{total_jobs}] [skip] {run_id} already has summary.json")

            if not ext_path.exists() and (rd / "samples.jsonl").exists():
                texts = [json.loads(line)["text"] for line in (rd / "samples.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
                ext = ni.score_extended_sample_metrics(texts=texts, ref_texts=ref_texts, ctw_depth=6, seed=int(eseed))
                ni.save_json_atomic(ext_path, ext)
                summ = ni.load_json(summ_path, {})
                summ.update(ext)
                ni.save_json_atomic(summ_path, summ)

    # --- STEP 3: AGGREGATE 5-SEED RESULTS ACROSS ALL 8 CONDITIONS & PACKAGE ZIP ---
    print("\n" + "=" * 118)
    print("=== PHASE 4 CONFIRMATORY RESULTS: ACTUAL ACE vs. ABLATIONS vs. RANK-1 SVD (SEEDS 201..205) ===")
    print("=" * 118)
    cond_records = {}
    for cond in ACE_EXPERIMENT_CONDITIONS:
        cid = cond["cond_id"]
        rows = []
        for eseed in EVAL_SEEDS:
            cand_ids = [f"p9_{cid}_seed{eseed}", f"p4_elfb_s32_g{float(cond['gamma'])}_sc{float(cond['sc_cfg'])}_seed{eseed}"]
            for rid in cand_ids:
                sp = PROJECT_ROOT / "runs" / rid / "summary.json"
                ep = PROJECT_ROOT / "runs" / rid / "extended_metrics.json"
                if sp.exists():
                    d = json.loads(sp.read_text(encoding="utf-8"))
                    if ep.exists():
                        d.update(json.loads(ep.read_text(encoding="utf-8")))
                    rows.append(d)
                    break
        cond_records[cid] = rows

    print(f"{'Condition':<26} {'n':>2} {'Gen-PPL':>14} {'H_ps (nats)':>16} {'rep-4':>16} {'JSD_pi':>15} {'Self-BLEU':>15} {'MAUVE':>14}")
    print("-" * 118)
    summary_out = {"calibration": {k: float(v) for k, v in cal_payload.items() if isinstance(v, (int, float))}, "conditions": {}}
    base_sc4_rows = cond_records.get("2_baseline_SC4", [])

    for cond in ACE_EXPERIMENT_CONDITIONS:
        cid = cond["cond_id"]
        rows = cond_records[cid]
        if not rows:
            continue
        def _ms(key):
            arr = np.array([float(r.get(key, np.nan)) for r in rows], dtype=float)
            return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0, arr

        ppl_m, ppl_s, ppl_a = _ms("gen_ppl")
        hps_m, hps_s, hps_a = _ms("entropy")
        rep_m, rep_s, rep_a = _ms("rep4")
        jsd_m, jsd_s, jsd_a = _ms("jsd_inter_sample")
        sb_m, sb_s, sb_a = _ms("self_bleu")
        mv_m, mv_s, mv_a = _ms("mauve")

        print(f"{cid:<26} {len(rows):>2} {ppl_m:>7.2f}+/-{ppl_s:<5.2f} {hps_m:>8.4f}+/-{hps_s:<6.4f} "
              f"{rep_m:>8.4f}+/-{rep_s:<6.4f} {jsd_m:>7.4f}+/-{jsd_s:<6.4f} {sb_m:>7.2f}+/-{sb_s:<6.2f} {mv_m:>6.3f}+/-{mv_s:<5.3f}")

        c_stats = {}
        for mkey in ["gen_ppl", "entropy", "rep4", "within_distinct2", "h_pool", "jsd_inter_sample", "distinct2", "self_bleu", "jcd_bits_per_byte", "mauve"]:
            m_val, s_val, arr = _ms(mkey)
            entry = {"mean": m_val, "sd": s_val}
            if base_sc4_rows and cid != "2_baseline_SC4":
                b_arr = np.array([float(r.get(mkey, np.nan)) for r in base_sc4_rows], dtype=float)
                if len(arr) == len(b_arr) and len(arr) > 1:
                    d_vec = arr - b_arr
                    diff = float(np.mean(d_vec))
                    se = float(np.std(d_vec, ddof=1) / math.sqrt(len(d_vec))) if np.std(d_vec, ddof=1) > 0 else 1e-12
                    t_ci = float(stats.t.ppf(0.975, df=len(d_vec) - 1))
                    p_val = float(2.0 * stats.t.sf(abs(diff) / se, df=len(d_vec) - 1))
                    entry.update({"diff_vs_SC4": diff, "ci95_vs_SC4": [diff - t_ci * se, diff + t_ci * se], "p_paired_vs_SC4": p_val})
            c_stats[mkey] = entry
        summary_out["conditions"][cid] = c_stats

    print("\n=== PAIRED DIFFERENCES VS. UNSTEERED SC=4.0 BASELINE (2_baseline_SC4) ===")
    for cid in ["3_actual_ACE_proj_a05", "4_actual_ACE_proj_a10", "5_ablation_ACE_sub_b02", "6_ablation_ACE_rand_orth", "7_ablation_anti_ACE", "8_heuristic_rank1_svd"]:
        if cid in summary_out["conditions"]:
            cs = summary_out["conditions"][cid]
            r4 = cs["rep4"]
            jp = cs["jsd_inter_sample"]
            ppl = cs["gen_ppl"]
            hps = cs["entropy"]
            if "diff_vs_SC4" in r4:
                print(f"  {cid:<26}: Delta_rep4 = {r4['diff_vs_SC4']:+.4f} (95% CI [{r4['ci95_vs_SC4'][0]:+.4f}, {r4['ci95_vs_SC4'][1]:+.4f}], p={r4['p_paired_vs_SC4']:.2e}) | "
                      f"Delta_JSD_pi = {jp['diff_vs_SC4']:+.4f} (p={jp['p_paired_vs_SC4']:.2e}) | Delta_PPL = {ppl['diff_vs_SC4']:+.2f} | Delta_Hps = {hps['diff_vs_SC4']:+.4f}")

    out_json_path = P9_DIR / "p9_ace_confirmatory_summary.json"
    out_json_path.write_text(json.dumps(summary_out, indent=2), encoding="utf-8")

    zip_out = PROJECT_ROOT / "runs" / "p9_ace_intervention_artifacts.zip"
    with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(out_json_path, arcname="runs/p9_ace_intervention/p9_ace_confirmatory_summary.json")
        if cal_pt_path.exists():
            zf.write(cal_pt_path, arcname="runs/p9_ace_intervention/ace_calibrated_directions.pt")
        for cond in ACE_EXPERIMENT_CONDITIONS:
            cid = cond["cond_id"]
            for eseed in EVAL_SEEDS:
                for fn in ["summary.json", "extended_metrics.json"]:
                    p = PROJECT_ROOT / "runs" / f"p9_{cid}_seed{eseed}" / fn
                    if p.exists():
                        zf.write(p, arcname=f"runs/p9_{cid}_seed{eseed}/{fn}")
    print(f"\nSaved ACE intervention archive to {zip_out}")
else:
    print("[STATUS: GPU EXECUTION PENDING] Attach a Colab A100 GPU to run calibration + evaluation.")

Configured 8 experimental conditions x 5 evaluation seeds (201..205) + 2 calibration seeds (301..302).
[STATUS: GPU EXECUTION PENDING] Attach a Colab A100 GPU to run calibration + evaluation.
